# Hand Gesture Classification Using MediaPipe Landmarks (HaGRID Dataset)

This notebook classifies hand gestures using 21 MediaPipe hand landmarks (x, y, z coordinates) extracted from the HaGRID dataset.  
We follow the full ML pipeline: **Data Loading → Visualization → Preprocessing → Model Training → Evaluation → Conclusion**.

## 0. Install & Import Dependencies

In [1]:
# Run this cell only in Google Colab
# !pip install mediapipe scikit-learn pandas numpy matplotlib seaborn opencv-python-headless -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import joblib

print('All libraries imported successfully.')

---
## 1. Data Loading

The CSV contains 21 hand landmarks per sample (x, y, z coordinates each) → **63 features** + 1 label column.

In [3]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# If running locally, adjust the path. In Colab, upload the file or mount Drive.
CSV_PATH = 'hand_landmarks_data.csv'   # change if needed

df = pd.read_csv(CSV_PATH)
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
df.head(3)

Dataset shape : (25675, 64)
Columns       : ['x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4', 'x5', 'y5', 'z5', 'x6', 'y6', 'z6', 'x7', 'y7', 'z7', 'x8', 'y8', 'z8', 'x9', 'y9', 'z9', 'x10', 'y10', 'z10', 'x11', 'y11', 'z11', 'x12', 'y12', 'z12', 'x13', 'y13', 'z13', 'x14', 'y14', 'z14', 'x15', 'y15', 'z15', 'x16', 'y16', 'z16', 'x17', 'y17', 'z17', 'x18', 'y18', 'z18', 'x19', 'y19', 'z19', 'x20', 'y20', 'z20', 'x21', 'y21', 'z21', 'label']


,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,...,x19,y19,z19,x20,y20,z20,x21,y21,z21,label
0,262.669968,257.304901,-3.649205e-07,257.417542,247.109055,0.004224,246.882957,241.716827,0.005798,236.384537,...,223.345093,255.490692,-0.020450,215.043365,258.114746,-0.024577,208.006393,259.608673,-0.026722,call
1,83.351778,346.059113,-2.345265e-07,81.925037,328.562347,-0.011102,90.080132,311.535248,-0.021096,95.641823,...,132.451618,341.794434,-0.038175,142.773582,342.829254,-0.037336,152.431698,343.015991,-0.036136,call
2,187.756977,260.235492,-2.417307e-07,195.460579,241.506035,-0.000184,207.259529,223.674339,-0.009687,215.413628,...,250.301010,268.602938,-0.044068,262.425133,271.276638,-0.040469,272.989952,272.272231,-0.038301,call


In [4]:
# Basic statistics
print('=== Data Types ===')
print(df.dtypes.value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\n=== Class Distribution ===')
print(df['label'].value_counts())

=== Data Types ===
float64    63
str         1
Name: count, dtype: int64

Missing values: 0

=== Class Distribution ===
label
three2             1653
palm               1649
four               1634
ok                 1592
stop_inverted      1567
call               1506
peace_inverted     1497
stop               1482
rock               1458
three              1456
peace              1441
like               1436
two_up_inverted    1371
two_up             1343
dislike            1295
one                1263
mute               1087
fist                945
Name: count, dtype: int64


In [ ]:
# Class distribution bar chart
fig_class_dist, ax = plt.subplots(figsize=(12, 4))
counts = df['label'].value_counts()
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Gesture Class')
ax.set_ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 2. Data Visualization

We plot the raw hand skeleton (21 keypoints connected along finger chains) for a few sample gestures.

In [6]:
# MediaPipe hand skeleton connections (0-indexed landmarks)
CONNECTIONS = [
    # Palm
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky
    (0, 17), (17, 18), (18, 19), (19, 20),
    # Knuckle connections
    (5, 9), (9, 13), (13, 17)
]

def plot_hand(ax, row, title=''):
    """Plot hand skeleton from a DataFrame row (raw coordinates)."""
    xs = [row[f'x{i+1}'] for i in range(21)]
    ys = [row[f'y{i+1}'] for i in range(21)]
    for s, e in CONNECTIONS:
        ax.plot([xs[s], xs[e]], [ys[s], ys[e]], 'g-', linewidth=1.5)
    ax.scatter(xs, ys, c='red', s=30, zorder=5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.invert_yaxis()   # image coordinates: y increases downward
    ax.axis('off')

print('Helper function defined.')

Helper function defined.


In [ ]:
# Plot one sample per gesture class
classes = df['label'].unique()
n_classes = len(classes)
ncols = 6
nrows = int(np.ceil(n_classes / ncols))

fig_raw_skeletons, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()

for idx, cls in enumerate(sorted(classes)):
    sample = df[df['label'] == cls].iloc[0]
    plot_hand(axes[idx], sample, title=cls)

# Hide unused axes
for j in range(idx + 1, len(axes)):
    axes[j].axis('off')

fig_raw_skeletons.suptitle('Hand Landmark Skeletons — One Sample per Gesture', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()